# 🎬 Birth of Vision — MNIST Digit Recognizer
### *From Pixels to Intelligence: How AI Learned to See Numbers*

**Dataset:** Kaggle MNIST — 42,000 samples · 784 pixel features · 10-class classification  
**Model:** XGBoost + PCA Feature Engineering  
**Validation Accuracy:** ~96.1%

---

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import xgboost as xgb

warnings.filterwarnings("ignore")
print("✅ Libraries loaded successfully.")
print(f"   XGBoost version : {xgb.__version__}")

## Step 1 — Load Dataset

In [ ]:
# ── Flexible path: works on Kaggle AND locally ──────────────────────
DATA_PATHS = [
    "/kaggle/input/digit-recognizer/train.csv",
    "/kaggle/input/competitions/digit-recognizer/train.csv",
    "train.csv",
]

df = None
for p in DATA_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"✅ Loaded from: {p}")
        break

if df is None:
    raise FileNotFoundError(
        "train.csv not found. Download from: "
        "https://www.kaggle.com/competitions/digit-recognizer/data"
    )

print(f"   Shape   : {df.shape}")
print(f"   Columns : label + {df.shape[1] - 1} pixel features")
df.head(3)

## Step 2 — Exploratory Data Analysis

In [ ]:
# ── Basic health check ──────────────────────────────────────────────
pixel_cols = [c for c in df.columns if c != "label"]
pixel_data = df[pixel_cols]

print("=== Dataset Health ===")
print(f"  Missing values  : {df.isnull().sum().sum()}")
print(f"  Duplicate rows  : {df.duplicated().sum()}")
print(f"\n=== Pixel Statistics ===")
print(f"  Min   : {pixel_data.values.min()}")
print(f"  Max   : {pixel_data.values.max()}")
print(f"  Mean  : {pixel_data.values.mean():.2f}")
print(f"  Std   : {pixel_data.values.std():.2f}")
print(f"  Zero-variance pixels: {(pixel_data.std() == 0).sum()}")

In [ ]:
# ── Label distribution + sample images ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

label_counts = df["label"].value_counts().sort_index()
axes[0].bar(label_counts.index, label_counts.values, color="steelblue", edgecolor="white")
axes[0].set_title("Label Distribution", fontsize=13)
axes[0].set_xlabel("Digit"); axes[0].set_ylabel("Count")
axes[0].set_xticks(range(10))
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 20, str(v), ha="center", fontsize=9)
axes[1].axis("off")
plt.tight_layout(); plt.show()

# One sample per digit
fig2, axes2 = plt.subplots(2, 5, figsize=(12, 5))
fig2.suptitle("Sample Images (one per digit)", fontsize=13)
for digit in range(10):
    sample = df[df["label"] == digit].iloc[0][pixel_cols].values.reshape(28, 28)
    ax = axes2[digit // 5][digit % 5]
    ax.imshow(sample, cmap="gray"); ax.set_title(f"Digit: {digit}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Pixel intensity distribution ────────────────────────────────────
plt.figure(figsize=(10, 3))
flat = pixel_data.values.flatten()
plt.hist(flat[flat > 0], bins=50, color="steelblue", edgecolor="none")
plt.title("Pixel Intensity Distribution (non-zero pixels)")
plt.xlabel("Pixel Value"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()

## Step 3 — Feature Engineering

In [ ]:
# ── 3a. Remove zero-variance pixels ─────────────────────────────────
X = df[pixel_cols].values.astype(np.float32)
y = df["label"].values

nonzero_mask = pixel_data.std() > 0
X = X[:, nonzero_mask.values]
print(f"[1] Zero-variance pixels removed → {X.shape[1]} features remaining")

# ── 3b. Normalize to [0, 1] ─────────────────────────────────────────
X = X / 255.0
print("[2] Normalized pixel values to [0, 1]")

# ── 3c. PCA dimensionality reduction ────────────────────────────────
N_COMPONENTS = 150
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X)
explained_var = pca.explained_variance_ratio_.cumsum()[-1]
print(f"[3] PCA → {N_COMPONENTS} components explain {explained_var * 100:.1f}% of variance")

# Explained variance plot
plt.figure(figsize=(10, 3))
cumvar = pca.explained_variance_ratio_.cumsum()
plt.plot(range(1, N_COMPONENTS + 1), cumvar * 100, color="steelblue")
plt.axhline(95, color="red", linestyle="--", label="95% threshold")
plt.xlabel("Number of Components"); plt.ylabel("Cumulative Explained Variance (%)")
plt.title("PCA Explained Variance"); plt.legend()
plt.tight_layout(); plt.show()

# ── 3d. Train / validation split ────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain : {X_train.shape[0]:,} samples")
print(f"Val   : {X_val.shape[0]:,} samples")

## Step 4 — Train XGBoost Model

In [ ]:
# ── XGBoost with improved hyperparameters ───────────────────────────
model = xgb.XGBClassifier(
    n_estimators       = 500,
    max_depth          = 7,
    learning_rate      = 0.1,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    min_child_weight   = 3,
    gamma              = 0.1,
    reg_alpha          = 0.05,
    reg_lambda         = 1.0,
    objective          = "multi:softmax",
    num_class          = 10,
    eval_metric        = "merror",
    early_stopping_rounds = 30,
    random_state       = 42,
    n_jobs             = -1,
    tree_method        = "hist",   # faster on CPU
    device             = "cpu",
)

model.fit(
    X_train, y_train,
    eval_set  = [(X_val, y_val)],
    verbose   = 50,
)

print(f"\n✅ Training complete. Best iteration: {model.best_iteration}")

## Step 5 — Evaluate

In [ ]:
# ── Metrics ─────────────────────────────────────────────────────────
y_pred = model.predict(X_val)
acc    = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {acc * 100:.4f}%\n")
print(classification_report(y_val, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# ── Confusion matrix ────────────────────────────────────────────────
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.title(f"Confusion Matrix  (Accuracy: {acc * 100:.2f}%)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

In [ ]:
# ── Training curve ──────────────────────────────────────────────────
results   = model.evals_result()
val_error = results["validation_0"]["merror"]

plt.figure(figsize=(10, 3))
plt.plot(val_error, color="steelblue")
plt.axvline(model.best_iteration, color="red", linestyle="--",
            label=f"Best round: {model.best_iteration}")
plt.title("XGBoost Validation Error Over Boosting Rounds")
plt.xlabel("Round"); plt.ylabel("Multi-class Error")
plt.legend(); plt.tight_layout(); plt.show()

## Step 6 — Save Model & Artifacts (for HuggingFace)

In [ ]:
# ── Save model + PCA + mask for HuggingFace deployment ──────────────
os.makedirs("hf_artifacts", exist_ok=True)

joblib.dump(model,        "hf_artifacts/xgb_model.pkl")
joblib.dump(pca,          "hf_artifacts/pca.pkl")
joblib.dump(nonzero_mask, "hf_artifacts/nonzero_mask.pkl")

print("✅ Saved to hf_artifacts/")
print("   ├── xgb_model.pkl")
print("   ├── pca.pkl")
print("   └── nonzero_mask.pkl")

# ── Kaggle submission CSV (test set) ────────────────────────────────
# Only runs if test.csv exists
test_paths = [
    "/kaggle/input/digit-recognizer/test.csv",
    "/kaggle/input/competitions/digit-recognizer/test.csv",
    "test.csv",
]
for p in test_paths:
    if os.path.exists(p):
        df_test  = pd.read_csv(p)
        X_test   = df_test.values.astype(np.float32)
        X_test   = X_test[:, nonzero_mask.values] / 255.0
        X_test   = pca.transform(X_test)
        y_test   = model.predict(X_test)
        sub      = pd.DataFrame({"ImageId": np.arange(1, len(y_test)+1), "Label": y_test.astype(int)})
        sub.to_csv("submission.csv", index=False)
        print(f"\n✅ submission.csv saved ({len(sub):,} rows)")
        break
else:
    print("\nℹ️  test.csv not found — skipping submission.csv generation")
    sub = pd.DataFrame({"ImageId": np.arange(1, len(y_val)+1), "Label": y_pred.astype(int)})
    sub.to_csv("submission.csv", index=False)
    print(f"   Used validation predictions instead ({len(sub):,} rows)")

print(f"\n🏆 Final Validation Accuracy: {acc * 100:.4f}%")
sub.head(10)